In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
sns.set_style("darkgrid")

In [2]:
markets = pd.read_parquet("../data/fpmms.parquet")

In [3]:
markets.columns

Index(['currentAnswer', 'id', 'title', 'market_creator'], dtype='object')

In [4]:
markets.head()

,currentAnswer,id,title,market_creator
0,Yes,0x0004c80810bffe4b686f7a61b0a43d62c1be91f7,Will any new public opinion poll regarding the...,quickstart
1,No,0x000a20e27ec193f3c4beab7b10d467654817d2b0,Will the United Nations approve the conversion...,quickstart
2,No,0x000f559a98ae762a31a2dfc5285cb0085d6246ee,Will the Pakistani government officially annou...,quickstart
3,No,0x0017cd58d6a7ee1451388c7d5b1051b4c0a041f5,Will the first floating offshore wind research...,quickstart
4,No,0x0020d13c89140b47e10db54cbd53852b90bc1391,Will the Francis Scott Key Bridge in Baltimore...,quickstart


In [5]:
markets.market_creator.value_counts()

market_creator
quickstart    6748
pearl          945
Name: count, dtype: int64

In [6]:
markets.id.nunique()

7693

# Getting a dataset of 300

In [9]:
qs_markets_sample = markets.iloc[:200]

In [13]:
pearl_markets_sample = markets.iloc[-100:]

In [14]:
pearl_markets_sample.market_creator.value_counts()

market_creator
pearl    100
Name: count, dtype: int64

In [15]:
markets_sample = pd.concat([qs_markets_sample, pearl_markets_sample], ignore_index=True)

In [16]:
markets_sample.market_creator.value_counts()

market_creator
quickstart    200
pearl         100
Name: count, dtype: int64

# Autocast dataset format

In [18]:
import pickle
import json

In [19]:
def prepare_questions(kwargs):
    test_questions = json.load(
        open("../data/autocast/autocast_questions_filtered.json")
    )
    with open("../data/autocast/autocast_questions_filtered.pkl", "rb") as f:
        url_to_content = pickle.load(f)
    num_questions = kwargs.pop("num_questions", len(test_questions))

    questions = []
    for q in test_questions:
        if q["qtype"] == "t/f" and q["answer"] is not None:
            questions.append(q)
        if len(questions) >= num_questions:
            break

    return questions, url_to_content

In [20]:
kwargs = {}
kwargs["num_questions"] = 20
# right now the only parameter that could be used is num_questions
# if not found by default it takes all questions from the dataset
questions, url_to_content = prepare_questions(kwargs)

In [22]:
questions[0]

{'question': 'Will the Export-Import Bank of the United States be re-authorized before 1 January 2016?',
 'id': 'G5',
 'background': "The Export-Import Bank's authorization expired on 1 July, but proponents of the bank are working to get it re-authorized (http://www.nytimes.com/2015/07/01/business/international/though-charter-is-expiring-export-import-bank-will-keep-its-doors-open.html , http://www.nytimes.com/2015/07/06/us/politics/us-export-import-bank-teetering-on-edge.html , http://thehill.com/policy/finance/247953-house-gop-draws-first-in-ex-im-showdown ). Legislation re-authorizing the bank must be signed into law by the President before taking effect.",
 'publish_time': 1441116141242,
 'close_time': '2015-12-04 14:00:25+00:00',
 'tags': ['Economic Policy', 'US Politics', 'US Policy'],
 'source_links': ['http://www.hartfordbusiness.com/article/20151005/NEWS01/310029963',
  'http://breakingdefense.com/2015/09/theyre-back-congress-likely-to-pass-short-term-budget-deal/',
  'http://

# Transform the markets dataset into questions list

In [23]:
def get_forecast_value(answer: str):
    if answer =="no":
        return 0.4
    return 0.7

In [24]:
def prepare_market_questions(markets_df: pd.DataFrame):
    """Adapting prediction markets for testing offline tools since no source links are provided"""
    questions = []
    num_markets = len(markets_df)
    for i in range(num_markets):
        row = markets_df.iloc[i]
        content = {}
        content["question"] = row["title"]
        content["answer"] = row["currentAnswer"].lower()
        content["crowd"] = [{"forecast": get_forecast_value(content["answer"])}]
        content["source_links"]=[]
        questions.append(content)

    return questions

In [25]:
new_questions = prepare_market_questions(markets_df=markets_sample)

In [26]:
new_questions[0]

{'question': 'Will any new public opinion poll regarding the Treaty Principles Bill be published in New Zealand by November 16, 2024?',
 'answer': 'yes',
 'crowd': [{'forecast': 0.7}],
 'source_links': []}